# CA1 - Retail Analytics and Recommendation System

This notebook contains the data preparation and analysis for Tasks 1 and 2 using only the allocated datasets. Task 3 business recommendations and Task 4 reflection are intentionally not written here.

Assigned stakeholder: **Operations Director**.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

data_path = Path.cwd()
retail = pd.read_csv(data_path / 'Retail_regions.csv')
customers = pd.read_csv(data_path / 'Customers_repeat.csv')
products = pd.read_csv(data_path / 'Products_sort.csv')
transactions = pd.read_csv(data_path / 'Transactions_made.csv')
datasets = {
    'Retail': retail,
    'Customers': customers,
    'Products': products,
    'Transactions': transactions,
}
print('Datasets loaded successfully.')

Datasets loaded successfully.


In [2]:
for name, data in datasets.items():
    print(f'\n{name}: {data.shape[0]:,} rows x {data.shape[1]} columns')
    display(data.head())


Retail: 5,100 rows x 14 columns


,StoreID,Region,StoreType,MarketingSpend,Footfall,OnlineTraffic,StaffHours,DiscountRate,CustomerSatisfaction,InventoryAvailability,CompetitorDistance,PreviousMonthSales,Season,SalesRevenue
0,1030,South,Retail Park,"5,216.03","1,360.67","15,143.65",274.44,24.25,84.01,83.19,19.07,"46,154.90",Summer,"148,983.86"
1,1013,North,Town,"2,589.26","1,759.54","9,575.30",354.44,33.88,76.32,86.92,15.91,"44,674.79",Autumn,"164,838.92"
2,1022,North,Retail Park,"5,236.47","1,444.46",NaN,343.29,37.86,79.49,90.43,9.48,"48,924.72",Summer,"152,132.29"
3,1094,North,Mall,"5,848.63","1,369.64","6,489.55",408.19,23.57,63.92,80.86,11.22,"52,203.35",Autumn,"165,743.22"
4,1037,North,Retail Park,"6,647.67","1,073.22","13,003.16",390.72,4.80,58.01,94.40,14.49,"33,369.48",Winter,"160,706.11"



Customers: 1,000 rows x 5 columns


,CustomerID,AgeGroup,Region,LoyaltyLevel,MembershipYears
0,1,18-25,North,Bronze,1
1,2,18-25,East,Bronze,2
2,3,50+,East,Bronze,5
3,4,36-50,East,Silver,12
4,5,26-35,East,Silver,13



Products: 200 rows x 6 columns


,ProductID,Category,Brand,Price,Rating,PopularityScore
0,1,Kitchen,FitLife,460.84,4.46,2
1,2,Electronics,TechMax,211.35,3.39,1
2,3,Home,HomePro,337.91,3.31,2
3,4,Gaming,TechMax,323.63,3.66,1
4,5,kitchen,PrimeGoods,202.09,2.52,4



Transactions: 18,711 rows x 5 columns


,CustomerID,ProductID,PurchaseCount,LastPurchaseDays,Spend
0,1,120,2,92,36.24
1,2,55,4,293,"1,872.16"
2,2,180,3,91,156.66
3,3,69,2,307,200.92
4,4,82,2,134,133.04


## Initial data-quality assessment

The following checks identify missing values, duplicate rows, data types, and summary statistics before any cleaning decisions are made.

In [3]:
quality_summary = []
for name, data in datasets.items():
    quality_summary.append({
        'Dataset': name,
        'Rows': len(data),
        'Columns': data.shape[1],
        'Missing cells': int(data.isna().sum().sum()),
        'Duplicate rows': int(data.duplicated().sum()),
    })
quality_summary = pd.DataFrame(quality_summary)
display(quality_summary)
for name, data in datasets.items():
    print(f'\n{name} data types')
    display(data.dtypes.to_frame('dtype'))
    print(f'{name} missing values')
    display(data.isna().sum().to_frame('missing_count'))
    print(f'{name} numeric summary')
    numeric_data = data.select_dtypes(include='number')
    display(numeric_data.describe().T)


,Dataset,Rows,Columns,Missing cells,Duplicate rows
0,Retail,5100,14,1428,100
1,Customers,1000,5,0,0
2,Products,200,6,15,0
3,Transactions,18711,5,0,393



Retail data types


,dtype
StoreID,int64
Region,str
StoreType,str
MarketingSpend,float64
Footfall,float64
OnlineTraffic,float64
StaffHours,float64
DiscountRate,float64
CustomerSatisfaction,float64
InventoryAvailability,float64


Retail missing values


,missing_count
StoreID,0
Region,0
StoreType,0
MarketingSpend,357
Footfall,0
OnlineTraffic,356
StaffHours,0
DiscountRate,0
CustomerSatisfaction,356
InventoryAvailability,359


Retail numeric summary


... (15 outputs left out)

Transactions missing values


,missing_count
CustomerID,0
ProductID,0
PurchaseCount,0
LastPurchaseDays,0
Spend,0


Transactions numeric summary


,count,mean,std,min,25%,50%,75%,max
CustomerID,"18,711.00",525.18,277.04,1.00,285.00,529.00,764.00,"1,000.00"
ProductID,"18,711.00",86.24,45.30,1.00,52.00,82.00,112.00,200.00
PurchaseCount,"18,711.00",2.51,1.12,1.00,2.00,3.00,4.00,4.00
LastPurchaseDays,"18,711.00",183.76,104.21,1.00,94.00,184.00,274.00,364.00
Spend,"18,711.00",589.70,495.87,16.39,200.92,410.32,884.50,"1,980.00"


In [4]:
print('Retail ID uniqueness:', retail['StoreID'].nunique(), 'unique values out of', len(retail), 'rows')
print('Customer ID uniqueness:', customers['CustomerID'].nunique(), 'unique values out of', len(customers), 'rows')
print('Product ID uniqueness:', products['ProductID'].nunique(), 'unique values out of', len(products), 'rows')
print('Transaction customer IDs missing from customer table:', (~transactions['CustomerID'].isin(customers['CustomerID'])).sum())
print('Transaction product IDs missing from product table:', (~transactions['ProductID'].isin(products['ProductID'])).sum())

Retail ID uniqueness: 100 unique values out of 5100 rows
Customer ID uniqueness: 1000 unique values out of 1000 rows
Product ID uniqueness: 200 unique values out of 200 rows
Transaction customer IDs missing from customer table: 0
Transaction product IDs missing from product table: 0


## Data cleaning and feature engineering

Exact duplicate rows are removed because they repeat the same observation. Missing numeric values are replaced with the relevant dataset median, which limits the influence of extreme values. No rows are removed solely because they are statistical outliers; instead, outliers are flagged for later investigation.

In [5]:
retail_clean = retail.copy()
customers_clean = customers.copy()
products_clean = products.copy()
transactions_clean = transactions.copy()

duplicate_counts = {
    'Retail': int(retail_clean.duplicated().sum()),
    'Customers': int(customers_clean.duplicated().sum()),
    'Products': int(products_clean.duplicated().sum()),
    'Transactions': int(transactions_clean.duplicated().sum()),
}

retail_clean = retail_clean.drop_duplicates().copy()
transactions_clean = transactions_clean.drop_duplicates().copy()

for column in retail_clean.select_dtypes(include='number').columns:
    retail_clean[column] = retail_clean[column].fillna(retail_clean[column].median())
products_clean['Rating'] = products_clean['Rating'].fillna(products_clean['Rating'].median())

cleaning_summary = pd.DataFrame({
    'Dataset': ['Retail', 'Customers', 'Products', 'Transactions'],
    'Original rows': [len(retail), len(customers), len(products), len(transactions)],
    'Clean rows': [len(retail_clean), len(customers_clean), len(products_clean), len(transactions_clean)],
    'Duplicate rows removed': [duplicate_counts['Retail'], 0, 0, duplicate_counts['Transactions']],
    'Missing cells after cleaning': [int(retail_clean.isna().sum().sum()), int(customers_clean.isna().sum().sum()), int(products_clean.isna().sum().sum()), int(transactions_clean.isna().sum().sum())],
})
display(cleaning_summary)


,Dataset,Original rows,Clean rows,Duplicate rows removed,Missing cells after cleaning
0,Retail,5100,5000,100,0
1,Customers,1000,1000,0,0
2,Products,200,200,0,0
3,Transactions,18711,18318,393,0


In [6]:
retail_clean['SalesGrowthRate'] = (
    (retail_clean['SalesRevenue'] - retail_clean['PreviousMonthSales'])
    / retail_clean['PreviousMonthSales']
)
retail_clean['RevenuePerVisitor'] = retail_clean['SalesRevenue'] / retail_clean['Footfall']
retail_clean['MarketingROI'] = retail_clean['SalesRevenue'] / retail_clean['MarketingSpend']
retail_clean['StaffHoursPerVisitor'] = retail_clean['StaffHours'] / retail_clean['Footfall']

engineered_features = [
    'SalesGrowthRate',
    'RevenuePerVisitor',
    'MarketingROI',
    'StaffHoursPerVisitor',
]
display(retail_clean[engineered_features].describe().T)
display(retail_clean[['SalesRevenue'] + engineered_features].head())

,count,mean,std,min,25%,50%,75%,max
SalesGrowthRate,"5,000.00",2.54,0.98,0.59,1.88,2.37,3.01,13.19
RevenuePerVisitor,"5,000.00",139.10,59.54,62.89,106.93,126.08,153.04,"1,277.29"
MarketingROI,"5,000.00",33.72,18.70,2.31,25.07,30.27,37.26,480.80
StaffHoursPerVisitor,"5,000.00",0.33,0.18,0.04,0.23,0.29,0.38,3.71


,SalesRevenue,SalesGrowthRate,RevenuePerVisitor,MarketingROI,StaffHoursPerVisitor
0,"148,983.86",2.23,109.49,28.56,0.20
1,"164,838.92",2.69,93.68,63.66,0.20
2,"152,132.29",2.11,105.32,29.05,0.24
3,"165,743.22",2.17,121.01,28.34,0.30
4,"160,706.11",3.82,149.74,24.17,0.36


The engineered features measure month-on-month sales change, revenue relative to footfall, sales generated per unit of marketing spend, and staffing intensity relative to visitors. These measures will support the later operations-focused analysis.